# 02 — Tiền xử lý: class mapping, chia theo video, crop 128×128 (S2-DATA)

Gọi `src/preprocessing.run_preprocessing` theo `configs/data_contract.yaml` (ADR-001): 22 lớp, gộp train+val gốc
rồi chia lại **theo video** 70/15/15 seed 42, crop bbox +10% lề, bỏ box < 8 px, resize thẳng 128×128 RGB.
Phụ trách: Hùng (dự phòng: Yến).

Kết quả: `<PROCESSED>/{train,val,test}/<class_name>/*.jpg`, `split_manifest.csv`, `class_split_counts.csv`,
`class_mapping.json`, `preprocessing_summary.json`. Trên Colab, bản `.zip` được lưu lên Drive để lần sau dùng lại.
**Không commit `data/`.**

> Notebook này **chỉ gọi hàm trong `src/`**: không `%%writefile`, không chép code nguồn vào cell, không token,
> không `git push`. Muốn sửa logic → sửa `src/` + `tests/` trên nhánh riêng và mở PR.
> Lưu notebook bằng *File → Save a copy in GitHub* vào **nhánh của mình**, xoá output trước khi commit.

## 1. Cài đặt

In [ ]:
# Cài đặt chung: Colab → mount Drive + clone repo; máy cá nhân → dùng repo đang mở.
import os, sys, subprocess, json, time
from pathlib import Path

BRANCH = "main"   # đổi thành "feat/s2-pipeline-end-to-end" nếu PR #21 chưa merge
REPO_URL = "https://github.com/phanyen06ym-ien/ocean-plastic-ml-starter.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/ocean-plastic-ml")   # thư mục chung của nhóm
    WORK = Path("/content")                                   # ổ đĩa Colab, đọc nhanh hơn Drive
    REPO = WORK / "repo"
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = REPO                                              # data/, runs/ đã nằm trong .gitignore
    WORK = REPO / "data"
os.chdir(REPO); sys.path.insert(0, str(REPO))
for sub in ["data/raw", "data/processed", "runs"]:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
print("Repo:", REPO, "| commit:",
      subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Dataset TrashCan gốc. Colab: giải nén .zip trên Drive ra ổ đĩa Colab. Máy cá nhân: giải nén sẵn vào data/raw/trashcan.
if IN_COLAB:
    RAW_ROOT = WORK / "trashcan"
    if not RAW_ROOT.exists():
        zips = sorted((DRIVE / "data/raw").glob("*.zip"))
        assert zips, f"Chưa có file .zip trong {DRIVE / 'data/raw'} — xem docs/colab_setup.md"
        RAW_ROOT.mkdir()
        subprocess.run(["unzip", "-q", str(zips[0]), "-d", str(RAW_ROOT)], check=True)
else:
    RAW_ROOT = WORK / "raw/trashcan"
assert RAW_ROOT.exists(), f"Không thấy dataset tại {RAW_ROOT}"
print("\n".join(str(p) for p in sorted(RAW_ROOT.rglob("instances_*_trashcan.json"))))

## 2. Chạy tiền xử lý

In [ ]:
import shutil
from src.common import load_contract
from src.preprocessing import run_preprocessing

FORCE = False   # True → xử lý lại dù đã có dữ liệu (ví dụ sau khi đổi data contract)
DATA_VERSION = load_contract()["dataset"]["data_version"]      # v1_crop128
PROCESSED = WORK / "processed" / DATA_VERSION
PROCESSED_ZIP = DRIVE / "data/processed" / f"{DATA_VERSION}.zip"

if IN_COLAB and PROCESSED_ZIP.exists() and not PROCESSED.exists() and not FORCE:
    PROCESSED.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", str(PROCESSED_ZIP), "-d", str(PROCESSED.parent)], check=True)
    print("Dùng lại", PROCESSED_ZIP)
elif FORCE or not (PROCESSED / "preprocessing_summary.json").exists():
    t0 = time.time()
    run_preprocessing(RAW_ROOT, PROCESSED, mapping_out=REPO / "configs/class_mapping.json")
    print(f"Xong sau {time.time() - t0:.0f}s")
    if IN_COLAB:
        subprocess.run(["zip", "-qr", str(PROCESSED_ZIP), DATA_VERSION], cwd=PROCESSED.parent, check=True)
        print("Đã lưu lên Drive:", PROCESSED_ZIP)

shutil.copy(PROCESSED / "class_mapping.json", REPO / "configs/class_mapping.json")
summary = json.loads((PROCESSED / "preprocessing_summary.json").read_text(encoding="utf-8"))

## 3. Kiểm tra bắt buộc trước khi huấn luyện

In [ ]:
import pandas as pd
from IPython.display import display
from src.preprocessing import check_no_leakage

manifest = pd.read_csv(PROCESSED / "split_manifest.csv")
check_no_leakage(manifest)
print("Không rò rỉ video/ảnh giữa các tập ✔")
print("Số video:", summary["num_videos"], "(Dataset Audit ghi nhận 312 — chênh nhiều thì kiểm tra video_id_regex)")
print("Trạng thái crop:", summary["crop_status"])
print("Crop theo tập:", summary["crops_per_split"])
print("Lớp thiếu ở val:", summary["classes_missing_in"]["val"])
print("Lớp thiếu ở test:", summary["classes_missing_in"]["test"])
display(pd.read_csv(PROCESSED / "class_split_counts.csv", index_col=0))

## 4. Bàn giao

- Ghi `DATA_VERSION`, đường dẫn Drive `data/processed/<DATA_VERSION>.zip` và các số ở mục 3 vào issue S2-DATA
  và `docs/dataset_report.md`.
- File được phép commit: `configs/class_mapping.json`. Không commit ảnh crop hay `split_manifest.csv`.